# LLM-as-Judge: Flip-Flop Analysis

This notebook analyzes the results of the LLM-as-Judge flip-flop detection pipeline.

**Experiment Design:**
- Judge model: Gemini 3 Flash (via OpenRouter)
- Target: Reasoning traces from Claude and Gemini moral reasoning experiments
- Goal: Detect position changes (flip-flopping) during model reasoning
- A "flip" = the model genuinely changes which answer it leans toward during reasoning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Style settings
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

## 1. Load Judge Results

In [ ]:
results_dir = Path('../results/processed')
checkpoint_dir = Path('../results/raw')

def load_judge_data(source, benchmark):
    """Load judge results, trying processed first then checkpoint."""
    prefix = 'judge_' if source == 'claude' else 'judge_gemini_'
    processed = results_dir / f'{prefix}{benchmark}_results.csv'
    checkpoint = checkpoint_dir / f'{prefix}{benchmark}_checkpoint.csv'
    
    for path in [processed, checkpoint]:
        if path.exists():
            return pd.read_csv(path)
    return None

# Load all judge results
judge_dfs = {}
for source in ['claude', 'gemini']:
    for benchmark in ['ethics', 'moralchoice', 'morables']:
        df = load_judge_data(source, benchmark)
        if df is not None:
            judge_dfs[f'{source}_{benchmark}'] = df

# Summary
print('Judge Results Summary')
print('=' * 50)
for key, df in judge_dfs.items():
    n_flips = df['flip_flop_detected'].sum() if 'flip_flop_detected' in df.columns else 0
    print(f'{key}: {len(df)} rows, {n_flips} flip-flops detected ({n_flips/len(df)*100:.1f}%)')

if not judge_dfs:
    print('\nNo judge results found. Run: python run_judge.py')

In [ ]:
# Combine all judge results into a single dataframe
if judge_dfs:
    all_judge = pd.concat(judge_dfs.values(), ignore_index=True)
    print(f'Total judge results: {len(all_judge)}')
    print(f'\nColumns: {list(all_judge.columns)}')
    print(f'\nParse errors: {all_judge["judge_parse_error"].sum() if "judge_parse_error" in all_judge.columns else "N/A"}')
    all_judge.head()

## 2. Overall Flip-Flop Rates

In [ ]:
if judge_dfs:
    # Filter to rows with valid judge output
    valid = all_judge[all_judge['flip_flop_detected'].notna()].copy()
    
    print('Flip-Flop Rates by Source and Benchmark')
    print('=' * 60)
    
    summary = valid.groupby(['source_model', 'benchmark']).agg(
        total=('flip_flop_detected', 'count'),
        n_flips=('flip_flop_detected', 'sum'),
        flip_rate=('flip_flop_detected', 'mean')
    ).reset_index()
    
    summary['flip_rate_pct'] = (summary['flip_rate'] * 100).round(1)
    print(summary.to_string(index=False))

In [ ]:
if judge_dfs:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    pivot = summary.pivot(index='benchmark', columns='source_model', values='flip_rate_pct')
    pivot.plot(kind='bar', ax=ax, width=0.7, edgecolor='black', linewidth=0.5)
    
    ax.set_ylabel('Flip-Flop Rate (%)')
    ax.set_xlabel('Benchmark')
    ax.set_title('Flip-Flop Rate by Benchmark and Model')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
    ax.legend(title='Source Model')
    
    # Add value labels
    for container in ax.containers:
        ax.bar_label(container, fmt='%.1f%%', padding=3)
    
    plt.tight_layout()
    plt.savefig('../outputs/judge_flip_flop_rates.png', dpi=150, bbox_inches='tight')
    plt.show()

## 3. Flip-Flop Rates by Prompt Level (Claude)

In [ ]:
# Claude-specific analysis by prompt level
claude_dfs = {k: v for k, v in judge_dfs.items() if 'claude' in k}

if claude_dfs:
    claude_all = pd.concat(claude_dfs.values(), ignore_index=True)
    valid_claude = claude_all[claude_all['flip_flop_detected'].notna()].copy()
    
    LEVEL_ORDER = [0, 2, 4, 5]
    LEVEL_LABELS = {0: 'L0\nDirect', 2: 'L2\nCoT', 4: 'L4\nDevil\'s\nAdvocate', 5: 'L5\nTwo-Pass'}
    
    print('Claude Flip-Flop Rate by Prompt Level')
    print('=' * 60)
    
    level_stats = valid_claude.groupby(['benchmark', 'level']).agg(
        total=('flip_flop_detected', 'count'),
        n_flips=('flip_flop_detected', 'sum'),
        flip_rate=('flip_flop_detected', 'mean')
    ).reset_index()
    
    level_stats['flip_pct'] = (level_stats['flip_rate'] * 100).round(1)
    print(level_stats.to_string(index=False))

In [ ]:
if claude_dfs:
    benchmarks = valid_claude['benchmark'].unique()
    n_benchmarks = len(benchmarks)
    
    fig, axes = plt.subplots(1, n_benchmarks, figsize=(5 * n_benchmarks, 5))
    if n_benchmarks == 1:
        axes = [axes]
    
    level_colors = {0: '#95a5a6', 2: '#3498db', 4: '#e74c3c', 5: '#9b59b6'}
    
    for ax, bm in zip(axes, sorted(benchmarks)):
        bm_data = level_stats[level_stats['benchmark'] == bm].copy()
        bm_data = bm_data[bm_data['level'].isin(LEVEL_ORDER)]
        bm_data = bm_data.sort_values('level')
        
        bars = ax.bar(
            range(len(bm_data)),
            bm_data['flip_pct'],
            color=[level_colors.get(l, 'gray') for l in bm_data['level']],
            edgecolor='black', linewidth=0.5
        )
        
        ax.set_xticks(range(len(bm_data)))
        ax.set_xticklabels([LEVEL_LABELS.get(l, str(l)) for l in bm_data['level']])
        ax.set_ylabel('Flip-Flop Rate (%)')
        ax.set_title(f'{bm.upper()}')
        
        for i, (_, row) in enumerate(bm_data.iterrows()):
            ax.text(i, row['flip_pct'] + 1, f"{row['flip_pct']:.1f}%",
                    ha='center', va='bottom', fontsize=10)
    
    fig.suptitle('Flip-Flop Rate by Prompt Level (Claude)', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.savefig('../outputs/judge_flip_flop_by_level.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. Flip-Flop Rates by Thinking Condition (Claude)

In [ ]:
if claude_dfs and 'thinking' in valid_claude.columns:
    print('Claude Flip-Flop Rate: Thinking Enabled vs Disabled')
    print('=' * 60)
    
    thinking_stats = valid_claude.groupby(['benchmark', 'thinking']).agg(
        total=('flip_flop_detected', 'count'),
        n_flips=('flip_flop_detected', 'sum'),
        flip_rate=('flip_flop_detected', 'mean')
    ).reset_index()
    
    thinking_stats['flip_pct'] = (thinking_stats['flip_rate'] * 100).round(1)
    print(thinking_stats.to_string(index=False))
    
    # Cross-tabulation: level x thinking
    print('\n\nFlip-Flop Rate by Level x Thinking:')
    cross = valid_claude.groupby(['level', 'thinking'])['flip_flop_detected'].mean().unstack()
    cross = (cross * 100).round(1)
    print(cross.to_string())

## 5. Flip-Flop Type Analysis (Beneficial vs Harmful)

In [ ]:
if judge_dfs:
    # Only look at rows where flip-flop was detected
    flipped = all_judge[all_judge['flip_flop_detected'] == True].copy()
    
    if len(flipped) > 0:
        print(f'Total flip-flops detected: {len(flipped)}')
        print(f'\nFlip-Flop Type Distribution:')
        print('=' * 40)
        
        type_counts = flipped['flip_flop_type'].value_counts()
        for ftype, count in type_counts.items():
            print(f'  {ftype}: {count} ({count/len(flipped)*100:.1f}%)')
        
        # By benchmark
        print(f'\nFlip-Flop Type by Benchmark:')
        print('=' * 60)
        type_by_bm = flipped.groupby(['benchmark', 'flip_flop_type']).size().unstack(fill_value=0)
        print(type_by_bm.to_string())
    else:
        print('No flip-flops detected.')

In [ ]:
if judge_dfs and len(flipped) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Pie chart of flip types
    type_colors = {'beneficial': '#27ae60', 'harmful': '#e74c3c', 'neutral': '#95a5a6', 'none': '#bdc3c7'}
    type_counts = flipped['flip_flop_type'].value_counts()
    axes[0].pie(
        type_counts.values,
        labels=type_counts.index,
        autopct='%1.1f%%',
        colors=[type_colors.get(t, 'gray') for t in type_counts.index],
        startangle=90
    )
    axes[0].set_title('Flip-Flop Type Distribution')
    
    # Stacked bar by benchmark
    type_by_bm = flipped.groupby(['benchmark', 'flip_flop_type']).size().unstack(fill_value=0)
    type_by_bm_pct = type_by_bm.div(type_by_bm.sum(axis=1), axis=0) * 100
    
    type_by_bm_pct.plot(
        kind='bar', stacked=True, ax=axes[1],
        color=[type_colors.get(c, 'gray') for c in type_by_bm_pct.columns],
        edgecolor='black', linewidth=0.5
    )
    axes[1].set_ylabel('Percentage')
    axes[1].set_xlabel('Benchmark')
    axes[1].set_title('Flip-Flop Type by Benchmark')
    axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=0)
    axes[1].legend(title='Type', bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()
    plt.savefig('../outputs/judge_flip_flop_types.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. Position Change Depth

In [ ]:
if judge_dfs and 'num_position_changes' in all_judge.columns:
    valid = all_judge[all_judge['num_position_changes'].notna()].copy()
    valid['num_position_changes'] = valid['num_position_changes'].astype(int)
    
    print('Position Change Distribution')
    print('=' * 40)
    change_counts = valid['num_position_changes'].value_counts().sort_index()
    for n, count in change_counts.items():
        print(f'  {n} changes: {count} ({count/len(valid)*100:.1f}%)')
    
    print(f'\nMean position changes: {valid["num_position_changes"].mean():.2f}')
    print(f'Max position changes: {valid["num_position_changes"].max()}')

In [ ]:
if judge_dfs and 'num_position_changes' in all_judge.columns:
    valid = all_judge[all_judge['num_position_changes'].notna()].copy()
    valid['num_position_changes'] = valid['num_position_changes'].astype(int)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Histogram of position changes
    axes[0].hist(valid['num_position_changes'], bins=range(0, valid['num_position_changes'].max() + 2),
                 edgecolor='black', linewidth=0.5, alpha=0.8, color='#3498db')
    axes[0].set_xlabel('Number of Position Changes')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Distribution of Position Changes')
    
    # Mean position changes by prompt level (Claude only)
    if 'level' in valid.columns:
        claude_valid = valid[valid['source_model'] == 'claude']
        if len(claude_valid) > 0:
            level_means = claude_valid.groupby('level')['num_position_changes'].mean()
            level_means = level_means.reindex([0, 2, 4, 5])
            
            level_colors = {0: '#95a5a6', 2: '#3498db', 4: '#e74c3c', 5: '#9b59b6'}
            bars = axes[1].bar(
                range(len(level_means)),
                level_means.values,
                color=[level_colors.get(l, 'gray') for l in level_means.index],
                edgecolor='black', linewidth=0.5
            )
            axes[1].set_xticks(range(len(level_means)))
            axes[1].set_xticklabels([f'L{l}' for l in level_means.index])
            axes[1].set_ylabel('Mean Position Changes')
            axes[1].set_title('Mean Position Changes by Prompt Level (Claude)')
            
            for i, v in enumerate(level_means.values):
                if not np.isnan(v):
                    axes[1].text(i, v + 0.02, f'{v:.2f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.savefig('../outputs/judge_position_changes.png', dpi=150, bbox_inches='tight')
    plt.show()

## 7. Flip-Flop and Correctness

In [ ]:
if judge_dfs and 'correct' in all_judge.columns:
    valid = all_judge[
        all_judge['flip_flop_detected'].notna() & 
        all_judge['correct'].notna()
    ].copy()
    
    if len(valid) > 0:
        print('Accuracy: Flip-Flop vs No Flip-Flop')
        print('=' * 50)
        
        acc_comparison = valid.groupby(['benchmark', 'flip_flop_detected']).agg(
            accuracy=('correct', 'mean'),
            n=('correct', 'count')
        ).reset_index()
        
        acc_comparison['accuracy_pct'] = (acc_comparison['accuracy'] * 100).round(1)
        print(acc_comparison.to_string(index=False))
        
        # Overall
        print(f'\nOverall:')
        for flipped_val in [False, True]:
            subset = valid[valid['flip_flop_detected'] == flipped_val]
            label = 'Flipped' if flipped_val else 'Stable'
            print(f'  {label}: {subset["correct"].mean()*100:.1f}% (n={len(subset)})')

In [ ]:
if judge_dfs and 'correct' in all_judge.columns:
    valid = all_judge[
        all_judge['flip_flop_detected'].notna() & 
        all_judge['correct'].notna()
    ].copy()
    
    if len(valid) > 0:
        fig, ax = plt.subplots(figsize=(10, 6))
        
        acc_pivot = valid.groupby(['benchmark', 'flip_flop_detected'])['correct'].mean().unstack() * 100
        acc_pivot.columns = ['Stable', 'Flipped']
        
        acc_pivot.plot(kind='bar', ax=ax, width=0.7,
                       color=['#27ae60', '#e74c3c'],
                       edgecolor='black', linewidth=0.5)
        
        ax.set_ylabel('Accuracy (%)')
        ax.set_xlabel('Benchmark')
        ax.set_title('Accuracy: Stable vs Flipped Responses')
        ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
        ax.legend(title='Reasoning')
        
        for container in ax.containers:
            ax.bar_label(container, fmt='%.1f%%', padding=3)
        
        plt.tight_layout()
        plt.savefig('../outputs/judge_accuracy_by_flip.png', dpi=150, bbox_inches='tight')
        plt.show()

## 8. Trajectory Analysis

In [ ]:
if judge_dfs and 'trajectory' in all_judge.columns:
    flipped = all_judge[all_judge['flip_flop_detected'] == True].copy()
    
    if len(flipped) > 0:
        print('Most Common Trajectories')
        print('=' * 50)
        
        traj_counts = flipped['trajectory'].value_counts().head(15)
        for traj, count in traj_counts.items():
            print(f'  {traj}: {count}')
        
        # Trajectory length distribution
        flipped['traj_length'] = flipped['trajectory'].str.count(',') + 1
        print(f'\nTrajectory Length Distribution:')
        print(flipped['traj_length'].describe().to_string())

## 9. Initial Lean vs Final Answer

In [ ]:
if judge_dfs and 'initial_lean' in all_judge.columns:
    flipped = all_judge[all_judge['flip_flop_detected'] == True].copy()
    
    if len(flipped) > 0:
        print('Initial Lean vs Final Answer (Flipped Items Only)')
        print('=' * 60)
        
        # Did the initial lean match the final answer?
        flipped['initial_matches_final'] = flipped['initial_lean'] == flipped['extracted_answer']
        
        print(f'Initial lean matches final answer: {flipped["initial_matches_final"].mean()*100:.1f}%')
        print(f'Initial lean differs from final answer: {(~flipped["initial_matches_final"]).mean()*100:.1f}%')
        
        # Among those where initial != final (genuine flips), was the flip correct?
        if 'correct' in flipped.columns:
            genuine_flips = flipped[~flipped['initial_matches_final']]
            genuine_flips_with_truth = genuine_flips[genuine_flips['correct'].notna()]
            
            if len(genuine_flips_with_truth) > 0:
                print(f'\nAmong genuine flips (initial != final, n={len(genuine_flips_with_truth)}):')
                print(f'  Final answer correct: {genuine_flips_with_truth["correct"].mean()*100:.1f}%')
                print(f'  Final answer wrong: {(~genuine_flips_with_truth["correct"].astype(bool)).mean()*100:.1f}%')

## 10. Judge Summary Examples

In [ ]:
if judge_dfs and 'judge_summary' in all_judge.columns:
    flipped = all_judge[all_judge['flip_flop_detected'] == True].copy()
    
    if len(flipped) > 0:
        # Show examples of beneficial flips
        beneficial = flipped[flipped['flip_flop_type'] == 'beneficial']
        if len(beneficial) > 0:
            print('BENEFICIAL FLIP-FLOP EXAMPLES')
            print('=' * 60)
            for _, row in beneficial.head(5).iterrows():
                print(f"\nItem: {row['item_id']} | Benchmark: {row['benchmark']}")
                print(f"Trajectory: {row['trajectory']}")
                print(f"Summary: {row['judge_summary']}")
                print('-' * 40)
        
        # Show examples of harmful flips
        harmful = flipped[flipped['flip_flop_type'] == 'harmful']
        if len(harmful) > 0:
            print('\n\nHARMFUL FLIP-FLOP EXAMPLES')
            print('=' * 60)
            for _, row in harmful.head(5).iterrows():
                print(f"\nItem: {row['item_id']} | Benchmark: {row['benchmark']}")
                print(f"Trajectory: {row['trajectory']}")
                print(f"Summary: {row['judge_summary']}")
                print('-' * 40)

## 11. Token Cost Analysis

In [ ]:
if judge_dfs:
    token_cols = ['judge_input_tokens', 'judge_output_tokens']
    available = [c for c in token_cols if c in all_judge.columns]
    
    if available:
        print('Judge Token Usage')
        print('=' * 50)
        
        for col in available:
            valid_tokens = all_judge[all_judge[col].notna()][col]
            print(f'\n{col}:')
            print(f'  Total: {valid_tokens.sum():,.0f}')
            print(f'  Mean: {valid_tokens.mean():.0f}')
            print(f'  Median: {valid_tokens.median():.0f}')
        
        total_input = all_judge['judge_input_tokens'].sum() if 'judge_input_tokens' in all_judge.columns else 0
        total_output = all_judge['judge_output_tokens'].sum() if 'judge_output_tokens' in all_judge.columns else 0
        
        print(f'\nEstimated cost (at ~$0.10/M input, ~$0.40/M output):')
        cost = (total_input * 0.10 / 1_000_000) + (total_output * 0.40 / 1_000_000)
        print(f'  ${cost:.2f}')

## 12. Key Findings

In [ ]:
if judge_dfs:
    valid = all_judge[all_judge['flip_flop_detected'].notna()].copy()
    flipped = valid[valid['flip_flop_detected'] == True]
    
    print('=' * 60)
    print('KEY FINDINGS')
    print('=' * 60)
    
    # 1. Overall flip rate
    overall_rate = valid['flip_flop_detected'].mean() * 100
    print(f'\n1. Overall flip-flop rate: {overall_rate:.1f}%')
    
    # 2. Most flip-prone benchmark
    bm_rates = valid.groupby('benchmark')['flip_flop_detected'].mean() * 100
    worst = bm_rates.idxmax()
    print(f'2. Most flip-prone benchmark: {worst} ({bm_rates[worst]:.1f}%)')
    
    # 3. Beneficial vs harmful
    if len(flipped) > 0 and 'flip_flop_type' in flipped.columns:
        n_beneficial = (flipped['flip_flop_type'] == 'beneficial').sum()
        n_harmful = (flipped['flip_flop_type'] == 'harmful').sum()
        print(f'3. Among flip-flops: {n_beneficial} beneficial, {n_harmful} harmful')
        if n_beneficial + n_harmful > 0:
            ratio = n_beneficial / (n_beneficial + n_harmful)
            print(f'   Beneficial ratio: {ratio:.1%}')
    
    # 4. Effect on accuracy
    if 'correct' in valid.columns:
        valid_with_truth = valid[valid['correct'].notna()]
        if len(valid_with_truth) > 0:
            acc_stable = valid_with_truth[valid_with_truth['flip_flop_detected'] == False]['correct'].mean()
            acc_flipped = valid_with_truth[valid_with_truth['flip_flop_detected'] == True]['correct'].mean()
            print(f'4. Accuracy stable: {acc_stable*100:.1f}%, flipped: {acc_flipped*100:.1f}%')
            print(f'   Difference: {(acc_flipped - acc_stable)*100:+.1f} percentage points')
    
    # 5. Prompt level with most flips (Claude)
    claude_valid = valid[valid['source_model'] == 'claude']
    if len(claude_valid) > 0 and 'level' in claude_valid.columns:
        level_rates = claude_valid.groupby('level')['flip_flop_detected'].mean() * 100
        most_flips = level_rates.idxmax()
        print(f'5. Most flip-prone prompt level (Claude): L{most_flips} ({level_rates[most_flips]:.1f}%)')